In [ ]:
# Imports

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, classification_report, roc_auc_score
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ADATBETÖLTÉS

def fetch_data(from_year=2015, to_year=2025):
    years = range(from_year, to_year+1)

    df = pd.DataFrame()
    for year in years:
        df_year = pd.read_excel(f"http://tennis-data.co.uk/{year}/{year}.xlsx")
        df_year["Year"] = year
        df = pd.concat([df, df_year], ignore_index=True)
    
    return df

df = fetch_data(2015, 2025)

# Szűrés csak az ATP meccsekre és érvényes dátumokra
df = df[df['ATP'] != '']  # ATP meccsek
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df = df.dropna(subset=['Date'])
df = df.sort_values('Date').reset_index(drop=True)

print(f"ATP meccsek érvényes dátummal: {len(df)}")

In [ ]:
# PLAYER1 ÉS PLAYER2 VÉLETLENSZERŰ HOZZÁRENDELÉSE

def assign_players(row):
    """
    Jobb rangsorú játékos legyen Player1
    Ha nincs rank adat, akkor odds alapján
    """
    # Rank alapú (alacsonyabb rank = jobb)
    has_rank = pd.notna(row['WRank']) and pd.notna(row['LRank'])
    
    if has_rank:
        if row['WRank'] < row['LRank']:
            # Winner a jobb rangsorú -> Player1
            return pd.Series({
                'Player1': row['Winner'],
                'Player2': row['Loser'],
                'Player1_Rank': row['WRank'],
                'Player2_Rank': row['LRank'],
                'Player1_Pts': row['WPts'],
                'Player2_Pts': row['LPts'],
                'Player1_AvgOdds': row['AvgW'],
                'Player2_AvgOdds': row['AvgL'],
                'Player1_Wsets': row['Wsets'],
                'Player2_Wsets': row['Lsets'],
                'Target': 1  # Player1 nyert
            })
        else:
            # Loser a jobb rangsorú -> Player1
            return pd.Series({
                'Player1': row['Loser'],
                'Player2': row['Winner'],
                'Player1_Rank': row['LRank'],
                'Player2_Rank': row['WRank'],
                'Player1_Pts': row['LPts'],
                'Player2_Pts': row['WPts'],
                'Player1_AvgOdds': row['AvgL'],
                'Player2_AvgOdds': row['AvgW'],
                'Player1_Wsets': row['Lsets'],
                'Player2_Wsets': row['Wsets'],
                'Target': 0  # Player1 vesztett
            })
    else:
        # Odds alapú (alacsonyabb odds = favorit)
        if row['AvgW'] <= row['AvgL']:
            # Winner a favorit -> Player1
            return pd.Series({
                'Player1': row['Winner'],
                'Player2': row['Loser'],
                'Player1_Rank': row['WRank'],
                'Player2_Rank': row['LRank'],
                'Player1_Pts': row['WPts'],
                'Player2_Pts': row['LPts'],
                'Player1_AvgOdds': row['AvgW'],
                'Player2_AvgOdds': row['AvgL'],
                'Player1_Wsets': row['Wsets'],
                'Player2_Wsets': row['Lsets'],
                'Target': 1
            })
        else:
            # Loser a favorit -> Player1
            return pd.Series({
                'Player1': row['Loser'],
                'Player2': row['Winner'],
                'Player1_Rank': row['LRank'],
                'Player2_Rank': row['WRank'],
                'Player1_Pts': row['LPts'],
                'Player2_Pts': row['WPts'],
                'Player1_AvgOdds': row['AvgL'],
                'Player2_AvgOdds': row['AvgW'],
                'Player1_Wsets': row['Lsets'],
                'Player2_Wsets': row['Wsets'],
                'Target': 0
            })

# Player assignment alkalmazása
print("Player assignment...")
player_data = df.apply(assign_players, axis=1)
df = pd.concat([df, player_data], axis=1)

print(f"Player1/Player2 oszlopok létrehozva")
print(f"Target eloszlás: {df['Target'].value_counts().to_dict()}")

In [ ]:
# FEATURE ENGINEERING - TÖRTÉNELMI ADATOK

def calculate_historical_stats(df):
    """Játékosonkénti történelmi statisztikák számítása - BŐVÍTETT VERZIÓ"""
    print("Bővített történelmi statisztikák számítása...")
    
    all_players = pd.concat([df['Player1'], df['Player2']]).unique()
    player_stats = {}
    
    print(f"Összes játékos: {len(all_players)}")
    
    for i, player in enumerate(all_players):
        if i % 100 == 0:
            print(f"Feldolgozás: {i}/{len(all_players)} játékos")
            
        player_matches = df[(df['Player1'] == player) | (df['Player2'] == player)].copy()
        player_matches = player_matches.sort_values('Date').reset_index(drop=True)
        
        stats = {
            # Alap statisztikák különböző időablakokra
            'matches_3': [], 'matches_5': [], 'matches_10': [], 'matches_20': [],
            'wins_3': [], 'wins_5': [], 'wins_10': [], 'wins_20': [],
            
            # Forma trend
            'form_trend': [],
            'recent_performance': [],
            
            # Pihenési napok
            'days_since_last': [],
            'avg_rest_days': [],
            
            # Felület statisztikák
            'surface_stats': {},
            
            # Head-to-head statisztikák (később töltjük fel)
            'h2h_stats': {},
            
            # Rangsor momentum
            'rank_momentum': [],
            'rank_trend_30d': [],
            
            # Set statisztikák
            'avg_sets_won': [],
            'avg_sets_lost': []
        }
        
        for idx, match in player_matches.iterrows():
            current_date = match['Date']
            
            # Előző meccsek kinyerése (csak a jelenlegi dátum előttiek)
            prev_matches = player_matches[player_matches['Date'] < current_date]
            
            # Alap win rate statisztikák különböző ablakokra
            for window in [3, 5, 10, 20]:
                window_matches = prev_matches.tail(window)
                total_matches = len(window_matches)
                
                if total_matches > 0:
                    wins = sum(1 for _, prev_match in window_matches.iterrows() 
                              if prev_match['Winner'] == player)
                    stats[f'matches_{window}'].append(total_matches)
                    stats[f'wins_{window}'].append(wins)
                else:
                    stats[f'matches_{window}'].append(0)
                    stats[f'wins_{window}'].append(0)
            
            # Forma trend számítása (utolsó 10 meccs súlyozott átlaga)
            if len(prev_matches) >= 5:
                recent_10 = prev_matches.tail(10)
                weights = np.exp(np.linspace(-1, 0, len(recent_10)))  # Exponenciális súlyok
                wins = [(1 if m['Winner'] == player else 0) for _, m in recent_10.iterrows()]
                form_trend = np.average(wins, weights=weights) if len(wins) > 0 else 0.5
                stats['form_trend'].append(form_trend)
                
                # Teljesítmény változása (utolsó 5 vs. előző 5 meccs)
                if len(prev_matches) >= 10:
                    last_5 = prev_matches.tail(5)
                    prev_5 = prev_matches.tail(10).head(5)
                    
                    last_5_wins = sum(1 for _, m in last_5.iterrows() if m['Winner'] == player) / 5
                    prev_5_wins = sum(1 for _, m in prev_5.iterrows() if m['Winner'] == player) / 5
                    
                    performance_change = last_5_wins - prev_5_wins
                    stats['recent_performance'].append(performance_change)
                else:
                    stats['recent_performance'].append(0)
            else:
                stats['form_trend'].append(0.5)
                stats['recent_performance'].append(0)
            
            # Days since last és átlagos pihenés
            if len(prev_matches) > 0:
                last_match_date = prev_matches.iloc[-1]['Date']
                days_since = (current_date - last_match_date).days
                stats['days_since_last'].append(days_since)
                
                # Átlagos pihenési napok az utolsó 5 meccsből
                if len(prev_matches) >= 2:
                    recent_matches = prev_matches.tail(5)
                    rest_days = []
                    for j in range(1, len(recent_matches)):
                        rest = (recent_matches.iloc[j]['Date'] - recent_matches.iloc[j-1]['Date']).days
                        rest_days.append(rest)
                    stats['avg_rest_days'].append(np.mean(rest_days) if rest_days else 7)
                else:
                    stats['avg_rest_days'].append(7)
            else:
                stats['days_since_last'].append(30)
                stats['avg_rest_days'].append(7)
            
            # Rangsor momentum (ha van WRank és LRank adat)
            player_rank = match['WRank'] if match['Winner'] == player else match['LRank']
            if len(prev_matches) >= 5 and pd.notna(player_rank):
                # Utolsó 5 meccs rangsor változása
                recent_ranks = []
                for _, prev_match in prev_matches.tail(5).iterrows():
                    rank = prev_match['WRank'] if prev_match['Winner'] == player else prev_match['LRank']
                    if pd.notna(rank):
                        recent_ranks.append(rank)
                
                if len(recent_ranks) >= 2:
                    rank_change = recent_ranks[0] - recent_ranks[-1]  # Pozitív = javulás
                    stats['rank_momentum'].append(rank_change)
                else:
                    stats['rank_momentum'].append(0)
            else:
                stats['rank_momentum'].append(0)
            
            # 30 napos rangsor trend
            thirty_days_ago = current_date - pd.Timedelta(days=30)
            month_matches = prev_matches[prev_matches['Date'] >= thirty_days_ago]
            if len(month_matches) >= 2:
                first_rank = None
                last_rank = None
                for _, m in month_matches.iterrows():
                    rank = m['WRank'] if m['Winner'] == player else m['LRank']
                    if pd.notna(rank):
                        if first_rank is None:
                            first_rank = rank
                        last_rank = rank
                
                if first_rank is not None and last_rank is not None:
                    rank_trend = first_rank - last_rank  # Pozitív = javulás
                    stats['rank_trend_30d'].append(rank_trend)
                else:
                    stats['rank_trend_30d'].append(0)
            else:
                stats['rank_trend_30d'].append(0)
            
            # Set statisztikák
            if len(prev_matches) >= 3:
                recent_sets = prev_matches.tail(10)
                sets_won = []
                sets_lost = []
                
                for _, prev_match in recent_sets.iterrows():
                    if player == prev_match['Winner']:
                        sets_won.append(prev_match['Wsets'])
                        sets_lost.append(prev_match['Lsets'])
                    else:
                        sets_won.append(prev_match['Lsets'])
                        sets_lost.append(prev_match['Wsets'])
                
                stats['avg_sets_won'].append(np.mean(sets_won) if sets_won else 1.5)
                stats['avg_sets_lost'].append(np.mean(sets_lost) if sets_lost else 1.5)
            else:
                stats['avg_sets_won'].append(1.5)
                stats['avg_sets_lost'].append(1.5)
            
            # Surface stats (mint előtte)
            surface = match['Surface']
            if surface not in stats['surface_stats']:
                surface_prev = prev_matches[prev_matches['Surface'] == surface]
                surface_wins = sum(1 for _, sm in surface_prev.iterrows() 
                                 if sm['Winner'] == player)
                surface_total = len(surface_prev)
                
                # Specialista faktorszámítás (surface win rate vs. összes win rate)
                overall_wins = sum(1 for _, pm in prev_matches.iterrows() 
                                 if pm['Winner'] == player)
                overall_total = len(prev_matches)
                overall_rate = overall_wins / overall_total if overall_total > 0 else 0.5
                surface_rate = surface_wins / surface_total if surface_total > 0 else 0.5
                
                specialist_factor = surface_rate - overall_rate
                
                stats['surface_stats'][surface] = {
                    'wins': surface_wins, 
                    'total': surface_total,
                    'win_rate': surface_rate,
                    'specialist_factor': specialist_factor
                }
        
        player_stats[player] = stats
    
    print(f"Bővített statisztikák számítása kész. Feldolgozva: {len(player_stats)} játékos")
    return player_stats

# Történelmi statisztikák számítása
player_stats = calculate_historical_stats(df)

In [ ]:
# H2H statisztikák

def calculate_head_to_head_stats(df, player_stats):
    """Head-to-head statisztikák számítása"""
    print("Head-to-head statisztikák számítása...")
    
    # Minden játékos párosra H2H statisztikákat számolunk
    for idx, match in df.iterrows():
        if idx % 2500 == 0:
            print(f"H2H feldolgozás: {idx}/{len(df)}")
            
        p1, p2 = match['Player1'], match['Player2']
        current_date = match['Date']
        
        if p1 in player_stats and p2 in player_stats:
            # P1 vs P2 korábbi meccsek
            h2h_matches = df[
                (((df['Player1'] == p1) & (df['Player2'] == p2)) |
                 ((df['Player1'] == p2) & (df['Player2'] == p1))) &
                (df['Date'] < current_date)
            ].sort_values('Date')
            
            h2h_total = len(h2h_matches)
            h2h_p1_wins = 0
            
            if h2h_total > 0:
                for _, h2h_match in h2h_matches.iterrows():
                    if h2h_match['Winner'] == p1:
                        h2h_p1_wins += 1
                
                # Utolsó H2H eredmény dátuma és eredménye
                last_h2h = h2h_matches.iloc[-1]
                days_since_last_h2h = (current_date - last_h2h['Date']).days
                last_h2h_winner = last_h2h['Winner']
                
                # Surface specifikus H2H
                surface_h2h = h2h_matches[h2h_matches['Surface'] == match['Surface']]
                surface_h2h_total = len(surface_h2h)
                surface_h2h_p1_wins = sum(1 for _, sm in surface_h2h.iterrows() 
                                        if sm['Winner'] == p1)
            else:
                days_since_last_h2h = 999
                last_h2h_winner = None
                surface_h2h_total = 0
                surface_h2h_p1_wins = 0
            
            # Statisztikák mentése mindkét játékosnak
            h2h_key = f"{p1}_vs_{p2}"
            
            player_stats[p1]['h2h_stats'][h2h_key] = {
                'total_matches': h2h_total,
                'wins': h2h_p1_wins,
                'win_rate': h2h_p1_wins / h2h_total if h2h_total > 0 else 0.5,
                'days_since_last': days_since_last_h2h,
                'last_winner': last_h2h_winner,
                'surface_total': surface_h2h_total,
                'surface_wins': surface_h2h_p1_wins,
                'surface_win_rate': surface_h2h_p1_wins / surface_h2h_total if surface_h2h_total > 0 else 0.5
            }
            
            player_stats[p2]['h2h_stats'][h2h_key] = {
                'total_matches': h2h_total,
                'wins': h2h_total - h2h_p1_wins,
                'win_rate': (h2h_total - h2h_p1_wins) / h2h_total if h2h_total > 0 else 0.5,
                'days_since_last': days_since_last_h2h,
                'last_winner': last_h2h_winner,
                'surface_total': surface_h2h_total,
                'surface_wins': surface_h2h_total - surface_h2h_p1_wins,
                'surface_win_rate': (surface_h2h_total - surface_h2h_p1_wins) / surface_h2h_total if surface_h2h_total > 0 else 0.5
            }
    
    print("Head-to-head statisztikák kész")
    return player_stats

# H2H statisztikák hozzáadása
player_stats = calculate_head_to_head_stats(df, player_stats)

In [ ]:
# FEATURE-ÖK HOZZÁADÁSA

def add_features(df, player_stats):
    """Feature-ök hozzáadása a dataframe-hez - JAVÍTOTT VERZIÓ"""
    print("Feature-ök hozzáadása...")
    
    # Alap feature-ök
    df['Rank_Diff'] = df['Player2_Rank'] - df['Player1_Rank']
    df['Pts_Diff'] = df['Player1_Pts'] - df['Player2_Pts']
    df['Odds_Diff'] = df['Player2_AvgOdds'] - df['Player1_AvgOdds']
    df['Implied_Prob1'] = 1 / df['Player1_AvgOdds']
    df['Implied_Prob2'] = 1 / df['Player2_AvgOdds']
    
    # Surface encoding
    le_surface = LabelEncoder()
    df['Surface_Encoded'] = le_surface.fit_transform(df['Surface'])
    
    # Round encoding  
    le_round = LabelEncoder()
    df['Round_Encoded'] = le_round.fit_transform(df['Round'])
    
    # Történelmi feature-ök hozzáadása
    valid_rows = []
    
    for idx, row in df.iterrows():
        if idx % 2500 == 0:
            print(f"Feldolgozás: {idx}/{len(df)} sor")
        p1, p2 = row['Player1'], row['Player2']
        
        # Csak akkor vesszük fel, ha mindkét játékosnak van statisztikája
        if p1 in player_stats and p2 in player_stats:
            stats_p1 = player_stats[p1]
            stats_p2 = player_stats[p2]
            
            # Megkeressük a játékos meccsét a statisztikákban
            player1_matches = df[(df['Player1'] == p1) | (df['Player2'] == p1)].copy()
            player1_matches = player1_matches.sort_values('Date')
            
            player2_matches = df[(df['Player1'] == p2) | (df['Player2'] == p2)].copy()
            player2_matches = player2_matches.sort_values('Date')
            
            # Keressük meg az aktuális meccs pozícióját a játékos meccslistájában
            try:
                p1_match_idx = player1_matches.index.get_loc(idx)
                p2_match_idx = player2_matches.index.get_loc(idx)
                
                # Ellenőrizzük, hogy van-e elég statisztika
                if (p1_match_idx < len(stats_p1['matches_3']) and 
                    p2_match_idx < len(stats_p2['matches_3'])):
                    
                    # Feature-ök kiszámítása
                    row_features = row.copy()
                    
                    # Player1 features
                    for window in [3, 5, 10, 20]:
                        wins = stats_p1[f'wins_{window}'][p1_match_idx]
                        matches = stats_p1[f'matches_{window}'][p1_match_idx]
                        row_features[f'P1_WinRate_{window}'] = wins / matches if matches > 0 else 0.5
                        
                        wins = stats_p2[f'wins_{window}'][p2_match_idx]
                        matches = stats_p2[f'matches_{window}'][p2_match_idx]
                        row_features[f'P2_WinRate_{window}'] = wins / matches if matches > 0 else 0.5

                    # Forma és momentum features
                    row_features['P1_FormTrend'] = stats_p1['form_trend'][p1_match_idx]
                    row_features['P2_FormTrend'] = stats_p2['form_trend'][p2_match_idx]
                    row_features['FormTrend_Diff'] = row_features['P1_FormTrend'] - row_features['P2_FormTrend']

                    row_features['P1_RecentPerformance'] = stats_p1['recent_performance'][p1_match_idx]
                    row_features['P2_RecentPerformance'] = stats_p2['recent_performance'][p2_match_idx]
                    row_features['RecentPerformance_Diff'] = row_features['P1_RecentPerformance'] - row_features['P2_RecentPerformance']

                    row_features['P1_RankMomentum'] = stats_p1['rank_momentum'][p1_match_idx]
                    row_features['P2_RankMomentum'] = stats_p2['rank_momentum'][p2_match_idx]
                    row_features['RankMomentum_Diff'] = row_features['P1_RankMomentum'] - row_features['P2_RankMomentum']

                    row_features['P1_RankTrend30d'] = stats_p1['rank_trend_30d'][p1_match_idx]
                    row_features['P2_RankTrend30d'] = stats_p2['rank_trend_30d'][p2_match_idx]
                    row_features['RankTrend30d_Diff'] = row_features['P1_RankTrend30d'] - row_features['P2_RankTrend30d']

                    # Pihenési features
                    row_features['P1_AvgRestDays'] = stats_p1['avg_rest_days'][p1_match_idx]
                    row_features['P2_AvgRestDays'] = stats_p2['avg_rest_days'][p2_match_idx]
                    row_features['AvgRestDays_Diff'] = row_features['P1_AvgRestDays'] - row_features['P2_AvgRestDays']

                    # Surface specialist features
                    surface = row['Surface']
                    p1_surface_stats = stats_p1['surface_stats'].get(surface, {'win_rate': 0.5, 'specialist_factor': 0})
                    p2_surface_stats = stats_p2['surface_stats'].get(surface, {'win_rate': 0.5, 'specialist_factor': 0})

                    row_features['P1_SurfaceWinRate'] = p1_surface_stats['win_rate']
                    row_features['P2_SurfaceWinRate'] = p2_surface_stats['win_rate']
                    row_features['SurfaceWinRate_Diff'] = row_features['P1_SurfaceWinRate'] - row_features['P2_SurfaceWinRate']

                    row_features['P1_SurfaceSpecialist'] = p1_surface_stats['specialist_factor']
                    row_features['P2_SurfaceSpecialist'] = p2_surface_stats['specialist_factor']
                    row_features['SurfaceSpecialist_Diff'] = row_features['P1_SurfaceSpecialist'] - row_features['P2_SurfaceSpecialist']

                    # Set statisztikák
                    row_features['P1_AvgSetsWon'] = stats_p1['avg_sets_won'][p1_match_idx]
                    row_features['P2_AvgSetsWon'] = stats_p2['avg_sets_won'][p2_match_idx]
                    row_features['AvgSetsWon_Diff'] = row_features['P1_AvgSetsWon'] - row_features['P2_AvgSetsWon']

                    row_features['P1_AvgSetsLost'] = stats_p1['avg_sets_lost'][p1_match_idx]
                    row_features['P2_AvgSetsLost'] = stats_p2['avg_sets_lost'][p2_match_idx]
                    row_features['AvgSetsLost_Diff'] = row_features['P1_AvgSetsLost'] - row_features['P2_AvgSetsLost']

                    # Head-to-Head features
                    h2h_key = f"{p1}_vs_{p2}"
                    h2h_stats = stats_p1['h2h_stats'].get(h2h_key, {
                        'total_matches': 0, 'win_rate': 0.5, 'days_since_last': 999,
                        'surface_total': 0, 'surface_win_rate': 0.5, 'last_winner': None
                    })

                    row_features['H2H_TotalMatches'] = h2h_stats['total_matches']
                    row_features['H2H_P1_WinRate'] = h2h_stats['win_rate']
                    row_features['H2H_DaysSinceLast'] = min(h2h_stats['days_since_last'], 999)
                    row_features['H2H_SurfaceMatches'] = h2h_stats['surface_total']
                    row_features['H2H_Surface_P1_WinRate'] = h2h_stats['surface_win_rate']
                    row_features['H2H_LastWinnerP1'] = 1 if h2h_stats['last_winner'] == p1 else 0 if h2h_stats['last_winner'] == p2 else 0.5

                    # További különbség features
                    row_features['WinRate20_Diff'] = row_features['P1_WinRate_20'] - row_features['P2_WinRate_20']
                    
                    valid_rows.append(row_features)
                    
            except (KeyError, ValueError):
                # Ha nem találjuk a meccset, kihagyjuk
                continue
    
    df_valid = pd.DataFrame(valid_rows)
    print(f"Érvényes meccsek: {len(df_valid)} / {len(df)}")
    print(f"Végső meccsek száma: {len(df_valid)}")
    return df_valid

df = add_features(df, player_stats)

In [ ]:
# FEATURE-ÖK KIVÁLASZTÁSA ÉS TRAIN/TEST SZETTVÁLASZTÁS

# Feature-ök kiválasztása
# Az add_features függvény feature_columns listájának bővítése:

feature_columns = [
    # Alap features
    'Rank_Diff', 'Pts_Diff', 'Odds_Diff', 'Implied_Prob1', 'Implied_Prob2',
    'Surface_Encoded', 'Round_Encoded', 'Best of',
    
    # Bővített win rate features
    'P1_WinRate_3', 'P1_WinRate_5', 'P1_WinRate_10', 'P1_WinRate_20',
    'P2_WinRate_3', 'P2_WinRate_5', 'P2_WinRate_10', 'P2_WinRate_20',
    
    # Forma és momentum features
    'P1_FormTrend', 'P2_FormTrend', 'FormTrend_Diff',
    'P1_RecentPerformance', 'P2_RecentPerformance', 'RecentPerformance_Diff',
    'P1_RankMomentum', 'P2_RankMomentum', 'RankMomentum_Diff',
    'P1_RankTrend30d', 'P2_RankTrend30d', 'RankTrend30d_Diff',
    
    # Pihenési features
    #'P1_DaysSinceLast', 'P2_DaysSinceLast', 'DaysSince_Diff',
    'P1_AvgRestDays', 'P2_AvgRestDays', 'AvgRestDays_Diff',
    
    # Surface features
    'P1_SurfaceWinRate', 'P2_SurfaceWinRate', 'SurfaceWinRate_Diff',
    'P1_SurfaceSpecialist', 'P2_SurfaceSpecialist', 'SurfaceSpecialist_Diff',
    
    # Set statisztikák
    'P1_AvgSetsWon', 'P2_AvgSetsWon', 'AvgSetsWon_Diff',
    'P1_AvgSetsLost', 'P2_AvgSetsLost', 'AvgSetsLost_Diff',
    
    # Head-to-Head features
    'H2H_TotalMatches', 'H2H_P1_WinRate', 'H2H_DaysSinceLast',
    'H2H_SurfaceMatches', 'H2H_Surface_P1_WinRate', 'H2H_LastWinnerP1',
    
    # Új különbség features
    #'WinRate3_Diff', 'WinRate5_Diff', 'WinRate10_Diff', 'WinRate20_Diff'
]

# Hiányzó értékek kezelése
df = df.dropna(subset=feature_columns + ['Target', 'Player1_AvgOdds'])

# 3-way split időbeli sorrendben
train_df = df[df['Year'] <= 2022].copy()      # Train: 2015-2022
val_df = df[df['Year'] == 2023].copy()        # Validation: 2023
test_df = df[df['Year'] >= 2024].copy()       # Test: 2024+

print(f"Train: {len(train_df)} meccs (2015-2022)")
print(f"Validation: {len(val_df)} meccs (2023)")
print(f"Test: {len(test_df)} meccs (2024+)")

X_train = train_df[feature_columns]
y_train = train_df['Target']
X_val = val_df[feature_columns]
y_val = val_df['Target']
X_test = test_df[feature_columns]
y_test = test_df['Target']

In [ ]:
# BASELINE MODELLEK - A modell értékének megértéséhez

print("=== BASELINE MODELLEK ===\n")

# Baseline 1: Mindig a favoritra fogadunk (alacsonyabb odds)
test_df['Favorite_Wins'] = (test_df['Player1_AvgOdds'] < test_df['Player2_AvgOdds']).astype(int)
baseline_favorite_acc = accuracy_score(test_df['Target'], test_df['Favorite_Wins'])
print(f"Baseline 1 (Always bet favorite): {baseline_favorite_acc:.4f}")

# Baseline 2: Mindig a jobb rangsorúra fogadunk
test_df['BetterRank_Wins'] = (test_df['Player1_Rank'] < test_df['Player2_Rank']).astype(int)
baseline_rank_acc = accuracy_score(test_df['Target'], test_df['BetterRank_Wins'])
print(f"Baseline 2 (Always bet better rank): {baseline_rank_acc:.4f}")

# Baseline 3: Random (50%)
print(f"Baseline 3 (Random guess): 0.5000")

print(f"\nA modellnek legalább {baseline_favorite_acc:.4f} accuracy-t kell elérnie!\n")

In [ ]:
# MODELL ODDS NÉLKÜL - Valódi prediktív erő tesztelése

print("=== MODELL ODDS NÉLKÜL ===\n")

# Features ODDS nélkül
features_no_odds = [f for f in feature_columns 
                    if 'Odds' not in f and 'Implied_Prob' not in f]

print(f"Features odds nélkül: {len(features_no_odds)}")
print(f"Eltávolított features: {[f for f in feature_columns if f not in features_no_odds]}\n")

# Train
X_train_no_odds = train_df[features_no_odds]
X_test_no_odds = test_df[features_no_odds]

model_no_odds = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42,
    class_weight='balanced'
)

model_no_odds.fit(X_train_no_odds, y_train)

# Predikció
y_pred_no_odds = model_no_odds.predict(X_test_no_odds)
y_pred_proba_no_odds = model_no_odds.predict_proba(X_test_no_odds)[:, 1]

acc_no_odds = accuracy_score(y_test, y_pred_no_odds)
auc_no_odds = roc_auc_score(y_test, y_pred_proba_no_odds)

print(f"Accuracy (no odds): {acc_no_odds:.4f}")
print(f"ROC AUC (no odds): {auc_no_odds:.4f}")

# Feature importance
feat_imp_no_odds = pd.DataFrame({
    'feature': features_no_odds,
    'importance': model_no_odds.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 10 legfontosabb features (odds nélkül):")
print(feat_imp_no_odds.head(10))

# ÖSSZEHASONLÍTÁS
print(f"\n=== ÖSSZEHASONLÍTÁS ===")
print(f"Baseline (favorite): {baseline_favorite_acc:.4f}")
print(f"Model no odds: {acc_no_odds:.4f}")
print(f"Difference: {acc_no_odds - baseline_favorite_acc:+.4f}")

if acc_no_odds < baseline_favorite_acc:
    print("\n⚠️  FIGYELEM: A modell ROSSZABB mint a baseline!")
    print("Ez azt jelenti, hogy a feature-ök nem hordoznak valódi prediktív információt.")
    print("A modell valószínűleg csak az odds-ot tanulja meg.")
else:
    print(f"\n✓ A modell jobb mint a baseline (+{(acc_no_odds - baseline_favorite_acc)*100:.2f}%)")

In [ ]:
# EGYSZERŰSÍTETT MODELL - TOP 6 FEATURE

import pickle
from datetime import datetime
import os

print("\n=== EGYSZERŰSÍTETT MODELL TOP 6 FEATURE-REL ===")

# TOP 6 FEATURE KIVÁLASZTÁSA ÉS MAGYARÁZATA
selected_features = {
    'Odds_Diff': 'Player2 átlagos odds - Player1 átlagos odds (pozitív = Player1 favorit)',
    'Implied_Prob1': 'Player1 implicit valószínűsége az odds alapján (1 / Player1_odds)',
    'Implied_Prob2': 'Player2 implicit valószínűsége az odds alapján (1 / Player2_odds)', 
    'H2H_P1_WinRate': 'Player1 győzelmi aránya a korábbi egymás elleni meccsekben',
    'H2H_LastWinnerP1': 'Utolsó egymás elleni meccs győztese (1=Player1, 0=Player2, 0.5=nincs adat)',
    'Rank_Diff': 'Player2 rangsor - Player1 rangsor (pozitív = Player1 jobb rangsorú)'
}

selected_feature_names = list(selected_features.keys())

print("Kiválasztott feature-ök:")
for i, (feature, explanation) in enumerate(selected_features.items(), 1):
    print(f"{i}. {feature}")
    print(f"   Jelentés: {explanation}")
    print()

# ADATOK ELŐKÉSZÍTÉSE
print("Adatok előkészítése a kiválasztott feature-ökkel...")

# Ellenőrizzük, hogy minden feature elérhető-e
available_features = []
for feature in selected_feature_names:
    if feature in df.columns:
        available_features.append(feature)
        print(f"✓ {feature} - elérhető")
    else:
        print(f"✗ {feature} - HIÁNYZIK!")

if len(available_features) != len(selected_feature_names):
    print(f"FIGYELEM: Csak {len(available_features)}/{len(selected_feature_names)} feature érhető el!")
    selected_feature_names = available_features

# Train/test adatok előkészítése
X_train_simple = train_df[selected_feature_names].copy()
y_train_simple = train_df['Target'].copy()
X_test_simple = test_df[selected_feature_names].copy()
y_test_simple = test_df['Target'].copy()

# Hiányzó értékek kezelése
print(f"\nHiányzó értékek train adatokban: {X_train_simple.isnull().sum().sum()}")
print(f"Hiányzó értékek test adatokban: {X_test_simple.isnull().sum().sum()}")

# Hiányzó értékek pótlása
X_train_simple = X_train_simple.fillna({
    'Odds_Diff': 0.0,
    'Implied_Prob1': 0.5,
    'Implied_Prob2': 0.5,
    'H2H_P1_WinRate': 0.5,
    'H2H_LastWinnerP1': 0.5,
    'Rank_Diff': 0.0
})

X_test_simple = X_test_simple.fillna({
    'Odds_Diff': 0.0,
    'Implied_Prob1': 0.5,
    'Implied_Prob2': 0.5,
    'H2H_P1_WinRate': 0.5,
    'H2H_LastWinnerP1': 0.5,
    'Rank_Diff': 0.0
})

print(f"\nVégső train adatok shape: {X_train_simple.shape}")
print(f"Végső test adatok shape: {X_test_simple.shape}")

# EGYSZERŰSÍTETT MODELL BETANÍTÁSA
print("\nEgyszerűsített modell betanítása...")

# Optimalizált hiperparaméterek az egyszerűbb modellhez
model_simple = RandomForestClassifier(
    n_estimators=200,           # Több fa, mivel kevesebb feature van
    max_depth=8,                # Nem túl mély, hogy elkerüljük az overfittinget
    min_samples_split=10,       # Konzervatív split
    min_samples_leaf=5,         # Konzervatív leaf size
    max_features='sqrt',        # Négyzetgyök a feature számból
    random_state=42,
    class_weight='balanced'     # Kiegyensúlyozott osztálysúlyok
)

# Betanítás
model_simple.fit(X_train_simple, y_train_simple)

# MODELL ÉRTÉKELÉSE
print("\n=== EGYSZERŰSÍTETT MODELL TELJESÍTMÉNYE ===")

# Predikciók
y_pred_proba_simple = model_simple.predict_proba(X_test_simple)[:, 1]
y_pred_simple = (y_pred_proba_simple > 0.5).astype(int)

# Alapmetrikák
accuracy_simple = accuracy_score(y_test_simple, y_pred_simple)
precision_simple = precision_score(y_test_simple, y_pred_simple)
recall_simple = recall_score(y_test_simple, y_pred_simple)
roc_auc_simple = roc_auc_score(y_test_simple, y_pred_proba_simple)

print(f"Accuracy: {accuracy_simple:.4f}")
print(f"Precision: {precision_simple:.4f}")
print(f"Recall: {recall_simple:.4f}")
print(f"ROC AUC: {roc_auc_simple:.4f}")

# Feature importance az egyszerűsített modellben
print("\n=== FEATURE IMPORTANCE (Egyszerűsített modell) ===")
feature_importance_simple = pd.DataFrame({
    'feature': selected_feature_names,
    'importance': model_simple.feature_importances_,
    'explanation': [selected_features[feat] for feat in selected_feature_names]
}).sort_values('importance', ascending=False)

for _, row in feature_importance_simple.iterrows():
    print(f"{row['feature']}: {row['importance']:.4f}")
    print(f"  → {row['explanation']}")
    print()

# BETTING SZIMULÁCIÓ
print("=== BETTING SZIMULÁCIÓ (Egyszerűsített modell) ===")

betting_results_simple, final_bankroll_simple, total_bets_simple, winning_bets_simple = kelly_betting_simulation(
    test_df, y_pred_proba_simple, max_kelly_fraction=0.05, min_edge=0.03
)

if total_bets_simple > 0:
    hit_rate_simple = winning_bets_simple / total_bets_simple
    roi_simple = ((final_bankroll_simple - 100) / 100) * 100
    
    print(f"Összes fogadás: {total_bets_simple}")
    print(f"Nyertes fogadások: {winning_bets_simple}")
    print(f"Hit Rate: {hit_rate_simple:.4f}")
    print(f"Végső bankroll: {final_bankroll_simple:.2f}")
    print(f"ROI: {roi_simple:.2f}%")
    
    if len(betting_results_simple) > 0:
        avg_edge = betting_results_simple['Edge'].mean()
        avg_kelly = betting_results_simple['Kelly_Fraction'].mean()
        print(f"Átlagos edge: {avg_edge:.4f}")
        print(f"Átlagos Kelly frakció: {avg_kelly:.4f}")
else:
    print("Nem voltak értékfogadási lehetőségek!")

# MODELL MENTÉSE
print("\n=== MODELL MENTÉSE ===")

# Models mappa létrehozása ha nem létezik
models_dir = "models"
os.makedirs(models_dir, exist_ok=True)

# Időbélyeg a fájlnévhez
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
model_filename = f"tennis_simple_model_{timestamp}.pkl"
model_path = os.path.join(models_dir, model_filename)

# Modell adatok összegyűjtése mentéshez
model_data = {
    'model': model_simple,
    'features': selected_features,
    'feature_names': selected_feature_names,
    'feature_importance': feature_importance_simple,
    'performance_metrics': {
        'accuracy': accuracy_simple,
        'precision': precision_simple,
        'recall': recall_simple,
        'roc_auc': roc_auc_simple
    },
    'betting_performance': {
        'total_bets': total_bets_simple,
        'winning_bets': winning_bets_simple,
        'hit_rate': hit_rate_simple if total_bets_simple > 0 else 0,
        'final_bankroll': final_bankroll_simple if total_bets_simple > 0 else 100,
        'roi': roi_simple if total_bets_simple > 0 else 0
    },
    'training_info': {
        'timestamp': timestamp,
        'train_samples': len(X_train_simple),
        'test_samples': len(X_test_simple),
        'model_params': model_simple.get_params()
    }
}

# Mentés
try:
    with open(model_path, 'wb') as f:
        pickle.dump(model_data, f)
    
    print(f"✓ Modell sikeresen mentve: {model_path}")
    print(f"  Fájlméret: {os.path.getsize(model_path) / 1024:.1f} KB")
    
    # Mentési információk
    print(f"\nMentett modell információi:")
    print(f"- Timestamp: {timestamp}")
    print(f"- Feature-ök száma: {len(selected_feature_names)}")
    print(f"- Training minták: {len(X_train_simple)}")
    print(f"- Test minták: {len(X_test_simple)}")
    print(f"- ROC AUC: {roc_auc_simple:.4f}")
    print(f"- Betting ROI: {roi_simple:.2f}%" if total_bets_simple > 0 else "- Betting ROI: N/A")

except Exception as e:
    print(f"✗ Hiba a modell mentésekor: {str(e)}")

In [ ]:
# MODELL BETANÍTÁSA

print("Modell betanítása...")
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=20,
    min_samples_leaf=10,
    random_state=42
)

model.fit(X_train, y_train)

# Feature importance
feature_importance = pd.DataFrame({
    'feature': feature_columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 10 legfontosabb feature:")
print(feature_importance.head(10))

In [ ]:
# PREDIKCIÓ ÉS ALAP METRIKÁK

y_pred_proba = model.predict_proba(X_test)[:, 1]
y_pred = (y_pred_proba > 0.5).astype(int)

print("\n=== ALAP METRIKÁK ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall: {recall_score(y_test, y_pred):.4f}")
print(f"ROC AUC: {roc_auc_score(y_test, y_pred_proba):.4f}")

In [ ]:
# BETTING SZIMULÁCIÓ

print("\n=== BETTING SZIMULÁCIÓ ===")

def betting_simulation(test_df, y_pred_proba, stake=1):
    """Betting szimuláció fix tétösszeggel"""
    
    results = []
    bankroll = 0
    bets_placed = 0
    bets_won = 0
    
    for idx, (_, match) in enumerate(test_df.iterrows()):
        pred_prob = y_pred_proba[idx]
        actual_result = match['Target']
        odds = match['Player1_AvgOdds']
        
        # Csak akkor fogadunk, ha van érték (value betting)
        implied_prob = 1 / odds
        if pred_prob > implied_prob:  # Value detected
            bets_placed += 1
            
            if actual_result == 1:  # Player1 nyert
                profit = (odds - 1) * stake
                bets_won += 1
                won = True
            else:  # Player1 vesztett
                profit = -stake
                won = False
            
            bankroll += profit
            hit = actual_result == 1
            
            results.append({
                'Match': idx,
                'Pred_Probability': pred_prob,
                'Implied_Probability': implied_prob,
                'Odds': odds,
                'Actual_Result': actual_result,
                'Profit': profit,
                'Bankroll': bankroll,
                'Hit': hit,
                'Value': pred_prob - implied_prob
            })
    
    return pd.DataFrame(results), bankroll, bets_placed, bets_won

# Szimuláció futtatása
betting_results, final_bankroll, total_bets, winning_bets = betting_simulation(test_df, y_pred_proba)

if len(betting_results) > 0:
    hit_rate = winning_bets / total_bets
    roi = (final_bankroll / (total_bets * 1)) * 100  # 1 egység tét
    
    print(f"Összes fogadás: {total_bets}")
    print(f"Nyertes fogadások: {winning_bets}")
    print(f"Hit Rate: {hit_rate:.4f}")
    print(f"Végső bankroll: {final_bankroll:.2f}")
    print(f"ROI: {roi:.2f}%")
    print(f"Profit faktor: {betting_results[betting_results['Profit'] > 0]['Profit'].sum() / abs(betting_results[betting_results['Profit'] < 0]['Profit'].sum()):.2f}")
else:
    print("Nem voltak értékfogadási lehetőségek!")

In [ ]:
# KELLY BETTING

def kelly_betting_simulation(test_df, y_pred_proba, 
                             initial_bankroll=100,
                             max_kelly_fraction=0.25,  # Fractional Kelly
                             min_edge=0.03,
                             min_confidence=0.55):
    """
    Javított Kelly betting szimuláció
    
    Parameters:
    - initial_bankroll: kezdő bankroll
    - max_kelly_fraction: max Kelly % (0.25 = quarter Kelly)
    - min_edge: minimum edge fogadáshoz
    - min_confidence: minimum predicted probability
    """
    
    results = []
    bankroll = initial_bankroll
    total_bets = 0
    winning_bets = 0
    
    for idx, (_, match) in enumerate(test_df.iterrows()):
        pred_prob = y_pred_proba[idx]
        actual_result = match['Target']
        odds = match['Player1_AvgOdds']
        
        # Csak akkor fogadunk, ha magabiztosak vagyunk
        if pred_prob < min_confidence:
            continue
        
        # Kelly számítás
        b = odds - 1
        p = pred_prob
        q = 1 - pred_prob
        
        # Edge számítás
        implied_prob = 1 / odds
        edge = p - implied_prob
        
        # Csak ha van jelentős edge
        if edge > min_edge:
            # Kelly formula: (bp - q) / b
            kelly_full = (b * p - q) / b
            
            # Fractional Kelly (konzervatívabb)
            kelly_fraction = max(0, min(kelly_full * max_kelly_fraction, max_kelly_fraction))
            
            if kelly_fraction > 0.01:  # Min 1% tét
                bet_amount = bankroll * kelly_fraction
                total_bets += 1
                
                if actual_result == 1:
                    profit = bet_amount * b
                    winning_bets += 1
                    won = True
                else:
                    profit = -bet_amount
                    won = False
                
                bankroll += profit
                
                # Ha a bankroll 20 alá esik, stop
                if bankroll < 20:
                    print(f"STOP: Bankroll túl alacsony ({bankroll:.2f}) bet #{total_bets} után")
                    break
                
                results.append({
                    'Match': idx,
                    'Pred_Probability': pred_prob,
                    'Edge': edge,
                    'Kelly_Full': kelly_full,
                    'Kelly_Fraction': kelly_fraction,
                    'Bet_Amount': bet_amount,
                    'Bet_Percent': (bet_amount/bankroll)*100,
                    'Odds': odds,
                    'Actual_Result': actual_result,
                    'Profit': profit,
                    'Bankroll': bankroll,
                    'Hit': won
                })
    
    return pd.DataFrame(results), bankroll, total_bets, winning_bets

# Használat
betting_results, final_bankroll, total_bets, winning_bets = kelly_betting_simulation(
    test_df, 
    y_pred_proba_simple,
    initial_bankroll=100,
    max_kelly_fraction=0.25,  # Quarter Kelly (konzervatív)
    min_edge=0.03,  # Min 3% edge
    min_confidence=0.55  # Min 55% predicted probability
)
print(f"Winning bets: {winning_bets}/{total_bets} ({winning_bets/total_bets:.1%})")
print(f"Final bankroll: {final_bankroll:.0f}")


In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# FEATURE-ÖK KIVÁLASZTÁSA ÉS PCA ALKALMAZÁSA

# Standard scaling és PCA a train/test split után
print("Feature scaling és PCA alkalmazása...")

# Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# PCA komponensek meghatározása - explained variance alapján
pca_full = PCA()
pca_full.fit(X_train_scaled)

# Cumulative explained variance
cumsum_var = np.cumsum(pca_full.explained_variance_ratio_)

# Komponensek száma 95% explained variance-hez
n_components_95 = np.argmax(cumsum_var >= 0.95) + 1
# Komponensek száma 99% explained variance-hez  
n_components_99 = np.argmax(cumsum_var >= 0.99) + 1

print(f"Eredeti feature-ök száma: {len(feature_columns)}")
print(f"95% variance-hoz szükséges komponensek: {n_components_95}")
print(f"99% variance-hoz szükséges komponensek: {n_components_99}")

# Több PCA verziót készítünk
pca_configs = {
    'pca_95': PCA(n_components=n_components_95),
    'pca_99': PCA(n_components=n_components_99),
    'pca_fixed_20': PCA(n_components=min(20, len(feature_columns))),
    'pca_fixed_10': PCA(n_components=min(10, len(feature_columns)))
}

pca_results = {}

for pca_name, pca_model in pca_configs.items():
    print(f"\n=== {pca_name.upper()} EREDMÉNYEK ===")
    
    # PCA fit és transform
    X_train_pca = pca_model.fit_transform(X_train_scaled)
    X_test_pca = pca_model.transform(X_test_scaled)
    
    print(f"Komponensek száma: {pca_model.n_components_}")
    print(f"Explained variance ratio: {pca_model.explained_variance_ratio_.sum():.4f}")
    
    # Modell betanítása PCA adatokon
    model_pca = RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        min_samples_split=20,
        min_samples_leaf=10,
        random_state=42
    )
    
    model_pca.fit(X_train_pca, y_train)
    
    # Predikció
    y_pred_proba_pca = model_pca.predict_proba(X_test_pca)[:, 1]
    y_pred_pca = (y_pred_proba_pca > 0.5).astype(int)
    
    # Metrikák
    accuracy_pca = accuracy_score(y_test, y_pred_pca)
    roc_auc_pca = roc_auc_score(y_test, y_pred_proba_pca)
    
    print(f"Accuracy: {accuracy_pca:.4f}")
    print(f"ROC AUC: {roc_auc_pca:.4f}")
    
    # Betting szimuláció
    betting_results_pca, final_bankroll_pca, total_bets_pca, winning_bets_pca = kelly_betting_simulation(
        test_df, y_pred_proba_pca
    )
    
    if total_bets_pca > 0:
        hit_rate_pca = winning_bets_pca / total_bets_pca
        roi_pca = ((final_bankroll_pca - 100) / 100) * 100
        
        print(f"Betting - Összes fogadás: {total_bets_pca}")
        print(f"Betting - Hit Rate: {hit_rate_pca:.4f}")
        print(f"Betting - ROI: {roi_pca:.2f}%")
        print(f"Betting - Final Bankroll: {final_bankroll_pca:.2f}")
    
    # Eredmények tárolása
    pca_results[pca_name] = {
        'model': model_pca,
        'pca': pca_model,
        'accuracy': accuracy_pca,
        'roc_auc': roc_auc_pca,
        'betting_results': betting_results_pca,
        'final_bankroll': final_bankroll_pca,
        'total_bets': total_bets_pca,
        'hit_rate': hit_rate_pca if total_bets_pca > 0 else 0,
        'roi': roi_pca if total_bets_pca > 0 else 0
    }

In [ ]:
# PLOTOK - VALÓSZÍNŰSÉGI BUCKETEK

print("\n=== VALÓSZÍNŰSÉGI ANALÍZIS ===")

def create_probability_buckets(betting_results, bucket_size=0.05):
    """Valószínűségi bucket-ek létrehozása és analízise"""
    
    betting_results['Prob_Bucket'] = (betting_results['Pred_Probability'] // bucket_size) * bucket_size
    bucket_stats = []
    
    for bucket in np.arange(0, 1, bucket_size):
        bucket_data = betting_results[
            (betting_results['Prob_Bucket'] >= bucket) & 
            (betting_results['Prob_Bucket'] < bucket + bucket_size)
        ]
        
        if len(bucket_data) > 0:
            actual_win_rate = bucket_data['Hit'].mean()
            avg_pred_prob = bucket_data['Pred_Probability'].mean()
            count = len(bucket_data)
            avg_odds = bucket_data['Odds'].mean()
            avg_value = bucket_data['Value'].mean()
            
            bucket_stats.append({
                'Predicted_Prob_Range': f"{bucket:.2f}-{bucket+bucket_size:.2f}",
                'Avg_Predicted_Prob': avg_pred_prob,
                'Actual_Win_Rate': actual_win_rate,
                'Count': count,
                'Avg_Odds': avg_odds,
                'Avg_Value': avg_value,
                'Calibration_Error': abs(avg_pred_prob - actual_win_rate)
            })
    
    return pd.DataFrame(bucket_stats)

if len(betting_results) > 0:
    bucket_stats = create_probability_buckets(betting_results)
    
    print("\nValószínűségi bucket statisztikák:")
    print(bucket_stats[['Predicted_Prob_Range', 'Avg_Predicted_Prob', 'Actual_Win_Rate', 'Count', 'Calibration_Error']])
    
    # 1. Plot: Predikció vs Tényleges win rate
    plt.figure(figsize=(15, 5))
    
    plt.subplot(1, 3, 1)
    plt.plot(bucket_stats['Avg_Predicted_Prob'], bucket_stats['Actual_Win_Rate'], 'bo-', label='Tényleges win rate')
    plt.plot([0, 1], [0, 1], 'r--', alpha=0.5, label='Ideális kalibráció')
    plt.xlabel('Átlagos prediktált valószínűség')
    plt.ylabel('Tényleges win rate')
    plt.title('Modell Kalibráció')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    # 2. Plot: Valószínűségi eloszlás
    plt.subplot(1, 3, 2)
    plt.bar(bucket_stats['Predicted_Prob_Range'], bucket_stats['Count'], alpha=0.7)
    plt.xlabel('Prediktált valószínűség tartomány')
    plt.ylabel('Fogadások száma')
    plt.title('Fogadások eloszlása valószínűség szerint')
    plt.xticks(rotation=45)
    
    # 3. Plot: Bankroll fejlődése
    plt.subplot(1, 3, 3)
    plt.plot(betting_results['Match'], betting_results['Bankroll'], 'g-', linewidth=2)
    plt.xlabel('Fogadás sorszáma')
    plt.ylabel('Bankroll')
    plt.title('Bankroll fejlődése')
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # 4. Plot: Value vs Hit Rate
    plt.figure(figsize=(10, 6))
    
    # Value bucket-ek
    betting_results['Value_Bucket'] = (betting_results['Value'] // 0.02) * 0.02
    value_stats = betting_results.groupby('Value_Bucket').agg({
        'Hit': ['mean', 'count'],
        'Profit': 'mean'
    }).round(3)
    
    value_stats.columns = ['Hit_Rate', 'Count', 'Avg_Profit']
    value_stats = value_stats.reset_index()
    value_stats = value_stats[value_stats['Count'] > 10]  # Csak jelentős bucket-ek
    
    plt.scatter(value_stats['Value_Bucket'], value_stats['Hit_Rate'], 
                s=value_stats['Count'], alpha=0.6, c=value_stats['Avg_Profit'], cmap='RdYlGn')
    plt.colorbar(label='Átlagos profit')
    plt.xlabel('Érték (Predikció - Implicit valószínűség)')
    plt.ylabel('Hit Rate')
    plt.title('Érték vs Hit Rate (méret = minta méret, szín = profit)')
    plt.grid(True, alpha=0.3)
    plt.axhline(y=0.5, color='r', linestyle='--', alpha=0.5)
    plt.axvline(x=0, color='r', linestyle='--', alpha=0.5)
    plt.show()

else:
    print("Nincs elég adat a plotokhoz!")

In [ ]:
# RÉSZLETES EREDMÉNYEK

if len(betting_results) > 0:
    print("\n=== TOP 10 LEGNAGYOBB ÉRTÉKŰ FOGADÁS ===")
    top_value_bets = betting_results.nlargest(10, 'Value')[['Pred_Probability', 'Implied_Probability', 'Value', 'Odds', 'Hit', 'Profit']]
    print(top_value_bets.round(4))
    
    print(f"\nÖsszefoglaló:")
    print(f"Összes fogadás: {total_bets}")
    print(f"Hit Rate: {hit_rate:.4f}")
    print(f"Végső profit: {final_bankroll:.2f} egység")
    print(f"ROI: {roi:.2f}%")

print("\n=== PROGRAM VÉGE ===")